In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

import numpy as np
import pandas as pd
import yaml
from pathlib import Path

import os


from methylseg import MethylSegPathway, MethylDataPrep, MethylStateAssignmentMethod, MethylationStates

In [3]:
ip = get_ipython()
path = None
if "__vsc_ipynb_file__" in ip.user_ns:
    path = ip.user_ns["__vsc_ipynb_file__"]

currfile = os.path.splitext(os.path.basename(path))[0]

REFERENCE_DIR = Path(currfile).resolve().parent.parent / Path("reference_files")

In [4]:
tcga_sample_info, tcga_sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "TCGA-BD-A3EP-01A_450k.tsv.gz",
    sample_id="TCGA-BD-A3EP-01A",
    resolution="450k",
    remove_low_coverage_like_cpgs=True,
).prepare()

In [5]:
hm450k_model = MethylSegPathway(
    train_sample_info=tcga_sample_info,
    out_dir="hm450k_model_outdir",
)

In [6]:
hm450k_model.fit_pathway(force_optimize_rules=True)

Random search: 100%|██████████| 500/500 [04:43<00:00,  1.77it/s]


In [7]:
hm450k_model.to_yaml(REFERENCE_DIR / "tcga_hm450k_model.yaml")

In [8]:
test_model = MethylSegPathway.from_yaml(REFERENCE_DIR / "tcga_hm450k_model.yaml")

In [9]:
wgbs_test_sample_info, wgbs_test_sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "WGBS_colon-primary-tumor_1_wgbs.tsv.gz",
    sample_id="colon-primary-tumor_1",
    resolution="wgbs",
    min_coverage=10,
    remove_low_coverage_like_cpgs=True,
).prepare()

In [10]:
wgbs_model = MethylSegPathway(
    train_sample_info=wgbs_test_sample_info,
    out_dir="wgbs_model_outdir",
    hmm_type="sticky"
)

In [11]:
wgbs_model.fit_pathway(force_optimize_rules=True)

Random search: 100%|██████████| 500/500 [13:27<00:00,  1.62s/it]


In [12]:
wgbs_model.to_yaml(REFERENCE_DIR / "wgbs_colon_model.yaml")

In [13]:
test_model = MethylSegPathway.from_yaml(REFERENCE_DIR / "tcga_hm450k_model.yaml")